# Basic OCR Box Extraction
Run EasyOCR detection, split oversized horizontal boxes, and score each split against the first split in its group.

In [ ]:
from pathlib import Path
import sys

import cv2

REPO_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists():
        REPO_ROOT = candidate
        break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.processing import (
    EasyOCRDetector,
    draw_boxes,
    split_big_boxes_and_score_similarity,
)

print(f"Repo root: {REPO_ROOT}")

Repo root: /home/tp_ubuntu/unsorted/handwriting-recognition


In [2]:
INPUT_IMAGE_PATH = REPO_ROOT / "data/raw/n-img.jpg"
OUTPUT_IMAGE_PATH = REPO_ROOT / "data/processed/easyocr/n-img_basic_split_boxes.jpg"

image_bgr = cv2.imread(str(INPUT_IMAGE_PATH))
if image_bgr is None:
    raise FileNotFoundError(f"Could not read image: {INPUT_IMAGE_PATH}")

detector = EasyOCRDetector(languages=["vi"], gpu=False)
detect_output = detector.detect_char_boxes(str(INPUT_IMAGE_PATH))

processed = split_big_boxes_and_score_similarity(
    image=image_bgr,
    detect_output=detect_output,
    width_ratio=1.8,
    vector_size=28,
)

annotated = draw_boxes(
    image=image_bgr.copy(),
    horizontal_boxes=processed["horizontal_list"],
    free_boxes=processed["free_list"],
    copy=False,
)

OUTPUT_IMAGE_PATH.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(OUTPUT_IMAGE_PATH), annotated)

print(f"Input image: {INPUT_IMAGE_PATH}")
print(f"Output image: {OUTPUT_IMAGE_PATH}")
print(f"Original horizontal boxes: {len(detect_output['horizontal_list'])}")
print(f"Split horizontal boxes: {len(processed['horizontal_list'])}")
print(f"Free boxes: {len(processed['free_list'])}")

Using CPU. Note: This module is much faster with a GPU.


Input image: /home/tp_ubuntu/unsorted/handwriting-recognition/data/raw/n-img.jpg
Output image: /home/tp_ubuntu/unsorted/handwriting-recognition/data/processed/easyocr/n-img_basic_split_boxes.jpg
Original horizontal boxes: 19
Split horizontal boxes: 36
Free boxes: 0


In [4]:
for group in processed["horizontal_groups"]:
    print(
        {
            "source_index": group["source_index"],
            "was_split": group["was_split"],
            "scores": [round(item["similarity_to_first"], 2) for item in group["items"]],
        }
    )

{'source_index': 0, 'was_split': False, 'scores': [100.0]}
{'source_index': 1, 'was_split': False, 'scores': [100.0]}
{'source_index': 2, 'was_split': False, 'scores': [100.0]}
{'source_index': 3, 'was_split': False, 'scores': [100.0]}
{'source_index': 4, 'was_split': False, 'scores': [100.0]}
{'source_index': 5, 'was_split': True, 'scores': [100.0, 99.49, 99.24]}
{'source_index': 6, 'was_split': False, 'scores': [100.0]}
{'source_index': 7, 'was_split': True, 'scores': [100.0, 99.29, 99.07]}
{'source_index': 8, 'was_split': False, 'scores': [100.0]}
{'source_index': 9, 'was_split': False, 'scores': [100.0]}
{'source_index': 10, 'was_split': False, 'scores': [100.0]}
{'source_index': 11, 'was_split': True, 'scores': [100.0, 98.92, 98.87, 99.32]}
{'source_index': 12, 'was_split': True, 'scores': [100.0, 99.6, 99.39]}
{'source_index': 13, 'was_split': True, 'scores': [100.0, 98.82, 98.83]}
{'source_index': 14, 'was_split': True, 'scores': [100.0, 99.1, 98.84, 98.87, 98.91, 99.17, 99.38]}